In [ ]:
import pandas as pd
import numpy as np
from castle.metrics import MetricsDAG
from dots import DOTS
from diffan import DiffAN
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
np.set_printoptions(precision=3)

# Load data

In [ ]:
data_path = './test_data'
samples_path = f'{data_path}/samples_lagged.csv'
time_series_samples_path = f'{data_path}/samples.csv'
true_causal_matrix_path = f'{data_path}/adjmat.csv'
time_series = pd.read_csv(time_series_samples_path)
X_full = pd.read_csv(samples_path)
true_causal_matrix = pd.read_csv(true_causal_matrix_path)
true_causal_matrix = true_causal_matrix.to_numpy()

nb_variables = time_series.shape[1]
nb_lags = X_full.shape[1] // nb_variables
print(f"Dataset size {X_full.shape[0]} nb_variables: {nb_variables}, nb_lags: {nb_lags}")

# Run DOTS (temporal) vs base DiffAN (static)

In [ ]:
nb_nodes = nb_lags*nb_variables
X = X_full.sample(min(997, len(X_full)))

dots_alg = DOTS(nb_nodes, masking = True, residue= False, learning_rate= 0.0001, nn_depth_mul= 3)
adj_dots = dots_alg.fit(X.to_numpy(), nb_timesteps=nb_lags, nb_variables=nb_variables)

In [ ]:
adj_dots

In [ ]:
pd.DataFrame(adj_dots, columns=X_full.columns, index=X_full.columns)

In [ ]:
diffan_alg = DiffAN(nb_nodes, masking = True, residue= False, learning_rate= 0.0001, nn_depth_mul= 3)
adj_diffan = diffan_alg.fit(X.to_numpy())

In [ ]:
adj_diffan

In [ ]:
metrics_temporal = MetricsDAG(adj_dots, B_true=true_causal_matrix).metrics
metrics_static = MetricsDAG(adj_diffan, B_true=true_causal_matrix).metrics

pd.DataFrame([metrics_temporal, metrics_static], index=['DOTS (temporal)', 'DiffAN (static)'])